# Test-Time Scaling for Small VLMs on Multilingual Exam QA

This notebook demonstrates the full test-time scaling (TTS) pipeline used in our
project for the ImageCLEF 2026 Multimodal Reasoning competition. We implement and
evaluate inference-time compute strategies for improving multimodal reasoning on
the EXAMS-V benchmark using Qwen2.5-VL-7B-Instruct.

The pipeline proceeds in three stages:
1. **Baselines** -- zero-shot, chain-of-thought, and self-consistency (majority vote)
2. **Search** -- PRM-BAS (beam-annealing search) over a describe-then-reason pipeline
3. **Verification** -- generative critic scoring and external PRM rescoring

All code imports from the `src/` package. Cells that require GPU and a running
vLLM server are guarded with a `RUN_INFERENCE` flag so the notebook remains
readable without hardware.

## 0. Setup

**Hardware requirements:** The inference cells below require a single GPU with at least
16 GB VRAM (e.g., NVIDIA A40). The model is loaded via vLLM in bfloat16. PRM rescoring
requires a second model (Qwen-VL-PRM-7B, ~14 GB) co-resident on the same GPU.

**Software:** Install dependencies with `pip install -r requirements.txt`. A vLLM
server is not needed; we use the in-process `vllm.LLM` engine directly.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

from datasets import load_dataset

# Toggle this to True on a GPU machine to run inference cells.
# When False, inference cells are skipped and the notebook serves as documentation.
RUN_INFERENCE = False

# Ensure project root is on the path so `src.*` imports resolve.
PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import Candidate, SelectionResult, extract_answer, normalize_answer_key

### Load the EXAMS-V dataset

EXAMS-V is a multilingual, multimodal multiple-choice QA benchmark covering 21
languages and 27 subjects. Each question has an image containing the question text
and answer options. We load the validation split from HuggingFace and select a
small demo subset of 5 questions for illustration.

In [ ]:
examsv = load_dataset("MBZUAI/EXAMS-V")
examsv_val = examsv["validation"]

print(f"Validation set: {examsv_val.num_rows} questions")
print(f"Columns: {examsv_val.column_names}")

# Select a small demo subset for illustration
DEMO_SIZE = 5
demo_items = [dict(examsv_val[i]) for i in range(DEMO_SIZE)]

for item in demo_items:
    print(f"  {item['sample_id'][:12]}  lang={item['language']:10s}  "
          f"subject={item['subject'][:20]:20s}  gold={item['answer_key']}")

### Load the model

We use `src.backend_vllm.load_model` to load Qwen2.5-VL-7B-Instruct into the
vLLM in-process engine. This returns `(llm, processor)` where `llm` is a
`vllm.LLM` instance and `processor` is the HuggingFace `AutoProcessor` used
for chat-template formatting.

In [ ]:
# Requires GPU and vLLM
if RUN_INFERENCE:
    from src.backend_vllm import load_model, generate, generate_n

    MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
    model, processor = load_model(
        MODEL_NAME,
        dtype="bfloat16",
        max_model_len=16384,
        gpu_memory_utilization=0.9,
    )
    print(f"Loaded {MODEL_NAME} via vLLM")

## 1. Baselines

We establish three baselines of increasing inference-time cost:

1. **Zero-shot** -- The model sees the exam image and is asked to output a single
   letter (A-E) using vLLM's `guided_choice` constraint. This eliminates parsing
   entirely: the model is forced to output exactly one token from the valid set.
2. **Chain-of-thought (CoT)** -- The model reasons step by step before committing
   to an answer. We use the MMMU-style closer ("End your answer with: Answer: X")
   which reduces parse-failure rate compared to the default closer.
3. **Self-consistency (N=8)** -- We sample N=8 independent CoT chains at temperature
   0.7, extract the answer from each, and take a majority vote.

### 1.1 Zero-shot with guided decoding

The simplest baseline: present the image and ask for a single letter. The
`guided_choice` parameter in `generate()` constrains the output vocabulary to
exactly `["A", "B", "C", "D"]` via vLLM's structured output support. This
matches the approach used by the official ImageCLEF baselines.

In [ ]:
ZERO_SHOT_PROMPT = (
    "Answer the following multiple-choice question based on the image. "
    "The question and answer options are shown in the image.\n\n"
    "Your response MUST be ONLY the single letter of the correct answer "
    "[A, B, C, D, E]. No explanation or other text."
)

# Requires GPU and vLLM
if RUN_INFERENCE:
    zero_shot_preds = {}
    for item in demo_items:
        out = generate(
            model, processor,
            prompt=ZERO_SHOT_PROMPT,
            image=item["image"],
            temperature=0.0,
            max_new_tokens=1,
            guided_choice=["A", "B", "C", "D"],
        )
        qid = item["sample_id"]
        gold = normalize_answer_key(item["answer_key"])
        zero_shot_preds[qid] = out.text.strip()
        print(f"  {qid[:12]}  pred={out.text.strip()}  gold={gold}  "
              f"correct={out.text.strip() == gold}")

    correct = sum(1 for qid, pred in zero_shot_preds.items()
                  if pred == normalize_answer_key(
                      next(it for it in demo_items if it["sample_id"] == qid)["answer_key"]))
    print(f"\nZero-shot accuracy: {correct}/{len(zero_shot_preds)}")

### 1.2 Chain-of-thought (CoT)

CoT prompting asks the model to reason step by step before answering. We use the
MMMU-style prompt closer ("End your answer with: Answer: X") which produces a
more reliably parseable output format than the default closer ("The answer is X").

The answer letter is extracted from the generated text using `extract_answer()`,
which matches patterns like "Answer: B", "the answer is C", or a bare letter.

In [ ]:
COT_PROMPT = (
    "Answer the following multiple-choice question based on the image. "
    "The question and answer options are shown in the image.\n\n"
    "Think step by step. First analyze the question, then reason through "
    "each option.\nEnd your answer with: Answer: X (where X is A, B, C, D, "
    "or E)."
)

# Requires GPU and vLLM
if RUN_INFERENCE:
    cot_preds = {}
    for item in demo_items:
        out = generate(
            model, processor,
            prompt=COT_PROMPT,
            image=item["image"],
            temperature=0.0,
            max_new_tokens=1024,
        )
        qid = item["sample_id"]
        gold = normalize_answer_key(item["answer_key"])
        pred = extract_answer(out.text)
        cot_preds[qid] = pred
        print(f"  {qid[:12]}  pred={pred}  gold={gold}  correct={pred == gold}")
        print(f"    Reasoning (truncated): {out.text[:120]}...")

    correct = sum(1 for qid, pred in cot_preds.items()
                  if pred == normalize_answer_key(
                      next(it for it in demo_items if it["sample_id"] == qid)["answer_key"]))
    print(f"\nCoT accuracy: {correct}/{len(cot_preds)}")

### 1.3 Self-consistency (N=8)

Self-consistency (Wang et al., 2023) samples N independent CoT chains at
temperature > 0, extracts the answer from each, and selects the majority answer.
This is a simple but effective test-time scaling strategy: more samples improve
accuracy by marginalizing over reasoning paths.

We use `generate_n()` to sample all N=8 chains in a single batched vLLM call.
This is substantially faster than N sequential calls because image encoding and
prompt prefill are paid only once. The answers are aggregated with
`majority_vote()` from `src.verify`, which handles tie-breaking by logprob.

In [ ]:
from src.verify import majority_vote

SC_N = 8  # number of sampled chains

# Requires GPU and vLLM
if RUN_INFERENCE:
    sc_preds = {}
    for item in demo_items:
        # Sample N chains in one batched call
        outputs = generate_n(
            model, processor,
            prompt=COT_PROMPT,
            n=SC_N,
            image=item["image"],
            temperature=0.7,
            max_new_tokens=1024,
            seed=42,
        )

        # Build Candidate objects for majority_vote
        candidates = []
        for i, out in enumerate(outputs):
            answer = extract_answer(out.text)
            candidates.append(Candidate(
                description_id=0, chain_id=i,
                description="", reasoning=out.text,
                answer=answer, logprob=out.logprob,
            ))

        # Majority vote
        result = majority_vote(candidates)
        qid = item["sample_id"]
        gold = normalize_answer_key(item["answer_key"])
        sc_preds[qid] = result.answer

        answers = [c.answer for c in candidates]
        print(f"  {qid[:12]}  votes={dict(Counter(answers))}  "
              f"selected={result.answer}  gold={gold}  "
              f"correct={result.answer == gold}  "
              f"confidence={result.confidence}")

    correct = sum(1 for qid, pred in sc_preds.items()
                  if pred == normalize_answer_key(
                      next(it for it in demo_items if it["sample_id"] == qid)["answer_key"]))
    print(f"\nSelf-consistency (N={SC_N}) accuracy: {correct}/{len(sc_preds)}")

## 2. Search Strategy: PRM-BAS (Beam-Annealing Search)

Our search strategy is based on PRM-BAS (Hu et al., 2025), adapted for the
multimodal setting. The key idea is to use a process reward model (PRM) to guide
a beam search over reasoning steps, pruning low-quality partial chains early
before they waste further compute.

### Describe-then-reason pipeline

Instead of generating CoT chains directly from the image (as in the baselines),
we use a two-stage pipeline inspired by MSA (Zhuang et al., 2025):

1. **Describe (Stage 1):** Generate N diverse image descriptions using the VLM.
   Each description captures the question text, answer options, and visual elements
   from the exam image. This is the only stage that uses the image.
2. **Reason (Stage 2):** For each description, generate M reasoning chains using
   text-only prompting. The reasoner receives only the caption text (no image),
   which decouples visual perception from logical reasoning.

This separation has two advantages: (a) text-only reasoning is cheaper and
allows higher sampling budgets, and (b) errors can be attributed to either the
perception stage (bad OCR) or the reasoning stage (bad logic).

### Beam-annealing search

PRM-BAS replaces the flat M-sample reasoning in Stage 2 with a guided beam
search. At each reasoning step:

- **Expand:** Each surviving beam generates B continuation steps.
- **Score:** A step scorer (PRM or generative critic) scores each continuation.
- **Anneal:** If the score spread among the top-W beams exceeds threshold tau,
  the beam width W is halved. This concentrates compute on promising paths.

Parameters:
- `B0` (initial beam width): number of first-step samples (default 8)
- `B` (expansion factor): continuations per beam per depth (default 4)
- `tau` (annealing threshold): score spread that triggers width reduction (default 0.05)
- `max_depth` (d): maximum number of reasoning steps (default 10)

### 2.1 Stage 1: Image description

`describe()` generates N diverse descriptions of the exam image at temperature
0.7. The prompt asks the model to extract question text, answer options, and
describe all visual elements (diagrams, graphs, tables, chemical structures).
All N samples are produced in a single batched vLLM call via `generate_n()`.

In [ ]:
from src.describe import describe, build_describe_prompt

# Show the describe prompt template
print("Describe prompt:")
print(build_describe_prompt("English"))

# Requires GPU and vLLM
if RUN_INFERENCE:
    item = demo_items[0]
    descriptions = describe(
        model, processor,
        image=item["image"],
        language=item["language"],
        n=4,
        temperature=0.7,
        max_tokens=1024,
        seed=42,
    )
    print(f"\nGenerated {len(descriptions)} descriptions")
    for i, d in enumerate(descriptions):
        print(f"\n--- Description {i} ({len(d.text)} chars) ---")
        print(d.text[:300] + "...")

### 2.2 Stage 2: Text-only reasoning with PRM-BAS

For each description, `prm_bas()` runs the beam-annealing search. The search
generates reasoning steps one at a time, scores each step with a step scorer,
and prunes the beam based on the scores.

The step scorer is a callable `(prior_steps, current_step) -> float` that scores
how promising a partial reasoning chain is. We provide two factories:

- `make_prm_step_scorer()`: Uses Qwen-VL-PRM-7B, an external discriminative PRM
  trained on process supervision data. One forward pass per step.
- `make_generative_step_scorer()`: Uses the policy model itself as a generative
  critic (training-free). More expensive (~3 generate calls per step) but
  requires no additional model.

Terminal beams are converted to `Candidate` objects via `beams_to_candidates()`
for downstream verification.

In [ ]:
from src.search import prm_bas, beams_to_candidates, make_prm_step_scorer, make_generative_step_scorer
from src.reason import build_reason_prompt

# Show the reasoning prompt template (MMMU-style CoT)
print("Reasoning prompt (MMMU-style CoT):")
print(build_reason_prompt("<<description text>>", use_cot=True, cot_style="mmmu"))

# Requires GPU and vLLM
if RUN_INFERENCE:
    # Use the generative critic as step scorer (no separate PRM model needed)
    step_scorer = make_generative_step_scorer(model, processor, image=item["image"])

    # Run PRM-BAS on the first description
    desc_text = descriptions[0].text
    beams, metadata = prm_bas(
        model, processor,
        description=desc_text,
        step_scorer=step_scorer,
        image=None,           # text-only reasoning (image was used in Stage 1)
        B0=4,                 # initial beam width
        B=2,                  # expansion factor
        anneal_tau=0.05,      # annealing threshold
        max_depth=6,          # max reasoning steps
        stop_marker="answer:",
        temperature=0.7,
        max_step_tokens=256,
        use_cot=True,
        cot_style="mmmu",
        seed=42,
    )

    print(f"\nPRM-BAS completed:")
    print(f"  Terminal beams: {metadata['n_terminals']}")
    print(f"  Width schedule: {metadata['width_schedule']}")
    print(f"  Max depth reached: {metadata['max_depth_reached']}")
    print(f"  Total scorer calls: {metadata['total_scorer_calls']}")
    print(f"  Terminal scores: {[f'{s:.3f}' for s in metadata['terminal_scores'][:5]]}")

    # Convert beams to Candidate objects
    candidates = beams_to_candidates(beams, description_id=0, description=desc_text)
    for c in candidates[:3]:
        print(f"\n  Candidate {c.uid}: answer={c.answer}, "
              f"reasoning={c.reasoning[:100]}...")

## 3. Verification and Scoring Strategies

After generating candidate reasoning chains (via flat sampling or PRM-BAS
search), we need to select the best answer. We implement two verification
strategies beyond majority vote:

1. **Generative critic** -- A training-free, prompted-only scorer that uses the
   policy VLM itself as a judge. Each chain is rated on three axes (step-intent,
   visual-alignment, logical-soundness) with a strict JSON rubric. Scores are
   mean-aggregated to a scalar in [0, 1].
2. **PRM rescoring** -- An external discriminative scorer (Qwen-VL-PRM-7B) that
   scores each reasoning step as correct (+) or incorrect (-). The per-step P(+)
   scores are averaged across steps to rank chains.

We also apply a **skip-on-high-confidence rule**: if majority vote agreement
reaches 5/8 or higher, we skip the more expensive rescoring and trust the
majority. This saves compute on easy questions where the model is already
confident.

### 3.1 Generative critic

The generative critic (inspired by GM-PRM, Zhang et al., 2025) uses the policy
model to evaluate reasoning chains without any additional trained model. For each
candidate chain, it prompts the VLM three times (once per evaluation axis):

- **step-intent:** Does each step address the question and build toward an answer?
- **visual-alignment:** Does the reasoning match what is visible in the image?
- **logical-soundness:** Does the final answer follow from valid logical steps?

Each axis produces an integer score (1-5) extracted via JSON parsing, mapped to
[0, 1] via `(score - 1) / 4`, then averaged across axes. The candidate with the
highest mean score is selected.

In [ ]:
from src.verify import (
    generative_critic_score,
    generative_critic_rank,
    GENERATIVE_CRITIC_AXES,
    GENERATIVE_CRITIC_PROMPT,
)

# Show the three evaluation axes
print("Generative critic axes:")
for axis_name, definition in GENERATIVE_CRITIC_AXES.items():
    print(f"\n  {axis_name}:")
    print(f"    {definition[:100]}...")

# Show the prompt template (for one axis)
print("\n\nCritic prompt template (truncated):")
print(GENERATIVE_CRITIC_PROMPT[:300] + "...")

# Requires GPU and vLLM
if RUN_INFERENCE:
    # Score a single candidate
    example_candidate = candidates[0]
    score = generative_critic_score(
        model, processor,
        image=item["image"],
        candidate=example_candidate,
        axes=["step-intent", "visual-alignment", "logical-soundness"],
        temperature=0.0,
        max_tokens=128,
    )
    print(f"\nGenerative critic score for candidate {example_candidate.uid}: {score:.3f}")

    # Rank all candidates by critic score
    critic_result = generative_critic_rank(
        candidates, model, processor,
        image=item["image"],
    )
    print(f"\nCritic-selected answer: {critic_result.answer}")
    print(f"Confidence: {critic_result.confidence}")
    print(f"Top score: {critic_result.metadata.get('top_score', 'N/A'):.3f}")

### 3.2 PRM rescoring with Qwen-VL-PRM-7B

Qwen-VL-PRM-7B (ob11/Qwen-VL-PRM-7B) is an external process reward model trained
on VL-PRM300K. It takes an image, a question, and a (partial) solution, then
predicts P(+) at the final token position, where + means "the solution so far is
on a correct path."

For each candidate chain, we split the reasoning into steps (on double-newline
boundaries) and score each step as a prefix of the full solution. The per-step
P(+) scores can be aggregated as:
- **step_mean:** Average P(+) across all steps (default).
- **step_min:** Minimum P(+) (pessimistic; catches chains that drift).
- **one_shot:** Score the entire chain in a single forward pass (cheaper).

The candidate with the highest aggregated score is selected.

**Skip-on-high-confidence:** When majority vote agreement is 5/8 or higher, we
skip PRM rescoring entirely. This reflects the observation that high-agreement
questions are typically easy, and rescoring rarely changes the answer but always
costs compute.

In [ ]:
from src.verify import (
    load_qwen_vl_prm,
    qwen_vl_prm_score,
    qwen_vl_prm_rank,
    _split_into_steps,
)

# Show how step splitting works
example_reasoning = (
    "The question asks about the capital of France.\n\n"
    "Looking at the options:\n"
    "A) London - capital of the UK\n"
    "B) Paris - capital of France\n\n"
    "Paris is the capital of France.\n\n"
    "Answer: B"
)
steps = _split_into_steps(example_reasoning)
print(f"Reasoning split into {len(steps)} steps:")
for i, step in enumerate(steps):
    print(f"  Step {i}: {step[:60]}...")

# Requires GPU (loads a second model ~14 GB)
if RUN_INFERENCE:
    prm_model, prm_processor, prm_tokenizer = load_qwen_vl_prm()
    print("Loaded Qwen-VL-PRM-7B")

    # Skip-on-high-confidence rule
    SKIP_THRESHOLD = 5  # out of 8 chains

    # Check majority vote agreement
    mv_result = majority_vote(candidates)
    valid_answers = [c.answer for c in candidates if c.answer is not None]
    top_count = Counter(valid_answers).most_common(1)[0][1] if valid_answers else 0

    if top_count >= SKIP_THRESHOLD:
        print(f"\nMajority agreement {top_count}/{len(valid_answers)} >= {SKIP_THRESHOLD}, "
              f"skipping PRM rescoring. Answer: {mv_result.answer}")
        prm_result = mv_result
    else:
        print(f"\nMajority agreement {top_count}/{len(valid_answers)} < {SKIP_THRESHOLD}, "
              f"running PRM rescoring...")

        # Score and rank all candidates with the PRM
        question_text = descriptions[0].text  # use description as question context
        prm_result = qwen_vl_prm_rank(
            candidates,
            image=item["image"],
            question=question_text,
            prm_model=prm_model,
            prm_processor=prm_processor,
            prm_tokenizer=prm_tokenizer,
            mode="step_mean",
        )
        print(f"PRM-selected answer: {prm_result.answer}")
        print(f"Confidence: {prm_result.confidence}")
        print(f"Top PRM score: {prm_result.metadata.get('top_score', 'N/A'):.3f}")

## 4. Evaluation and Analysis

We evaluate predictions using stratified accuracy: overall accuracy broken down by
language, subject, and content type. This reveals where the model succeeds and
where it struggles (e.g., non-Latin scripts, STEM subjects with diagrams).

The analysis pipeline operates on the JSONL records produced by the full pipeline
run. Each record contains all descriptions, reasoning chains, and the verifier
decision for a single question. The post-hoc analysis script
(`scripts/analyze.py`) computes:
- Stratified accuracy by language, subject, and content type
- (N, M) scaling grid: accuracy as a function of the number of descriptions (N)
  and chains per description (M), obtained by subsampling the full candidate set
- Chain agreement statistics
- Parse-failure rates

### 4.1 Stratified accuracy computation

Given a set of predictions and gold labels, we compute accuracy per stratum
(language, subject). This is essential for understanding model behavior on a
multilingual benchmark like EXAMS-V, where performance can vary widely across
languages (e.g., English vs. Chinese vs. Serbian).

In [ ]:
from collections import defaultdict


def compute_stratified_accuracy(records, stratify_by="language"):
    """Compute accuracy stratified by a metadata field.

    Args:
        records: list of dicts with "predicted", "gold", and stratify_by fields.
        stratify_by: metadata field name to group by.

    Returns:
        dict mapping stratum name to (correct, total, accuracy).
    """
    buckets = defaultdict(lambda: [0, 0])
    for r in records:
        gold = r.get("gold")
        pred = r.get("predicted")
        if gold is None:
            continue
        key = r.get(stratify_by, "unknown")
        buckets[key][1] += 1
        if pred == gold:
            buckets[key][0] += 1

    results = {}
    for key, (correct, total) in sorted(buckets.items()):
        results[key] = {
            "correct": correct,
            "total": total,
            "accuracy": correct / total if total > 0 else 0.0,
        }
    return results


# Demonstrate on the demo subset (using zero-shot predictions if available)
if RUN_INFERENCE:
    demo_records = []
    preds_to_use = zero_shot_preds  # or sc_preds, cot_preds
    for item in demo_items:
        qid = item["sample_id"]
        demo_records.append({
            "predicted": preds_to_use.get(qid),
            "gold": normalize_answer_key(item["answer_key"]),
            "language": item["language"],
            "subject": item["subject"],
        })

    print("Stratified by language:")
    for lang, stats in compute_stratified_accuracy(demo_records, "language").items():
        print(f"  {lang:15s}  {stats['correct']}/{stats['total']}  "
              f"acc={stats['accuracy']:.2f}")

    print("\nStratified by subject:")
    for subj, stats in compute_stratified_accuracy(demo_records, "subject").items():
        print(f"  {subj:25s}  {stats['correct']}/{stats['total']}  "
              f"acc={stats['accuracy']:.2f}")

### 4.2 Chain agreement and parse-failure analysis

Two additional diagnostics:

- **Chain agreement:** For self-consistency and DTR, we track how many of the N
  chains agree on the same answer. High agreement (e.g., 7/8 or 8/8) indicates
  an easy question where the model is confident. Low agreement (e.g., 3/8) signals
  genuine ambiguity or difficulty.
- **Parse failures:** Questions where `extract_answer()` returns `None` for all
  chains. These represent a total failure to produce a parseable answer letter,
  typically caused by truncation at `max_tokens` before the model emits an
  answer. Parse-failure rates vary by language (e.g., Chinese questions with
  verbose reasoning are more prone to truncation).

For full post-hoc analysis on a completed run, use:
```bash
python scripts/analyze.py --run-dir runs/<run_id>
```
This computes the (N, M) scaling grid, stratified accuracies, and writes a
`summary.json` file.

In [ ]:
def analyze_chain_agreement(candidates_per_question):
    """Analyze chain agreement across questions.

    Args:
        candidates_per_question: dict mapping question_id to list of Candidate.

    Returns:
        dict with agreement statistics.
    """
    agreement_counts = []
    parse_fail_count = 0

    for qid, candidates in candidates_per_question.items():
        answers = [c.answer for c in candidates if c.answer is not None]
        if not answers:
            parse_fail_count += 1
            continue
        top_count = Counter(answers).most_common(1)[0][1]
        agreement_counts.append(top_count)

    total = len(candidates_per_question)
    return {
        "total_questions": total,
        "parse_failures": parse_fail_count,
        "parse_failure_rate": parse_fail_count / total if total else 0.0,
        "mean_agreement": sum(agreement_counts) / len(agreement_counts) if agreement_counts else 0.0,
        "agreement_distribution": dict(Counter(agreement_counts)),
    }


# Demonstrate with the self-consistency candidates (if available)
if RUN_INFERENCE:
    # sc_candidates_per_question would be populated from the SC loop above
    print("Chain agreement analysis would be shown here for a full run.")

## 5. Guided Parse Repair

A persistent failure mode in self-consistency and DTR is parse failure: the model
generates a long reasoning chain that hits `max_tokens` before emitting an answer
letter. When all N chains for a question fail to parse, the question receives no
prediction and counts as incorrect.

Our guided parse repair (`scripts/repair_parse_failures.py`) eliminates this
problem using a two-phase approach:

1. For each question where all chains failed to parse, load the original image.
2. For each failed chain, append an answer suffix (`"\n\nAnswer: "`) to the
   existing reasoning text.
3. Run a single guided-decoding token (vLLM `guided_choice=["A","B","C","D","E"]`,
   `max_tokens=1`) to force the model to commit to a letter, conditioned on
   its own reasoning.
4. Majority-vote the forced answers across chains.

This is equivalent to the two-phase generation in `backend_vllm.generate_two_phase()`.
By conditioning on the full reasoning context, the forced letter reflects the model's
actual conclusion. vLLM's automatic prefix caching means the Phase 2 prompt (which
shares the entire Phase 1 prefix) has near-zero marginal prefill cost.

This approach eliminates parse failures by construction: every question is
guaranteed to produce a valid answer letter.

In [ ]:
from src.backend_vllm import generate_two_phase

# Requires GPU and vLLM
if RUN_INFERENCE:
    item = demo_items[0]

    # Demonstrate two-phase generation: free reasoning + forced answer
    out = generate_two_phase(
        model, processor,
        prompt=COT_PROMPT,
        image=item["image"],
        temperature=0.7,
        max_new_tokens=1024,
        seed=42,
        answer_suffix="\n\nAnswer: ",
        answer_choices=["A", "B", "C", "D"],
    )

    # The output always ends with a valid answer letter
    gold = normalize_answer_key(item["answer_key"])
    pred = extract_answer(out.text)
    print(f"Two-phase output (last 100 chars): ...{out.text[-100:]}")
    print(f"Extracted answer: {pred}  gold={gold}  correct={pred == gold}")
    print(f"Tokens: prompt={out.prompt_tokens}, completion={out.completion_tokens}")
    print(f"Latency: {out.latency_s:.2f}s")

In [ ]:
def repair_parse_failures(candidates_per_question, model, processor, dataset_items):
    """Repair all-unparseable questions using guided decoding.

    For questions where every chain failed to extract an answer, appends
    "Answer: " to each chain's reasoning and forces a single guided-choice
    token. This is the core logic of scripts/repair_parse_failures.py.

    Args:
        candidates_per_question: dict mapping qid to list of Candidate.
        model: vLLM LLM instance.
        processor: HF AutoProcessor.
        dataset_items: dict mapping qid to dataset row (with "image" field).

    Returns:
        dict mapping repaired qids to their forced majority-vote answers.
    """
    from src.backend_vllm import _build_prompt_inputs, _build_sampling_params

    ANSWER_SUFFIX = "\n\nAnswer: "
    ANSWER_CHOICES = ["A", "B", "C", "D", "E"]

    repaired = {}
    for qid, candidates in candidates_per_question.items():
        # Check if all chains failed
        if any(c.answer is not None for c in candidates):
            continue

        item = dataset_items.get(qid)
        if item is None:
            continue

        # Build guided decoding inputs for each chain
        base_inputs = _build_prompt_inputs(processor, COT_PROMPT, item["image"])
        p2_inputs = []
        for c in candidates:
            p2_text = base_inputs["prompt"] + c.reasoning + ANSWER_SUFFIX
            p2_inputs.append({
                "prompt": p2_text,
                "multi_modal_data": {"image": item["image"]},
            })

        sp = _build_sampling_params(
            temperature=0.0,
            max_new_tokens=1,
            n=1,
            guided_choice=ANSWER_CHOICES,
        )

        outputs = model.generate(p2_inputs, sp)
        forced_answers = [out.outputs[0].text.strip() for out in outputs]
        winner = Counter(forced_answers).most_common(1)[0][0]
        repaired[qid] = winner

    return repaired


# The full repair script is run via:
# python scripts/repair_parse_failures.py --candidates runs/<run_id>/candidates.jsonl
print("Parse repair function defined. For a full run, use:")
print("  python scripts/repair_parse_failures.py --candidates runs/<run_id>/candidates.jsonl")